# MvDEC from-scratch two-view arm (S2), 10 seeds - REUTERS

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Table 2 reports a *best performance*; this runs the from-scratch S2 protocol for 10 consecutive seeds, one process per seed and reports the whole distribution (mean, std, median, best) next to the paper row. Best-of-N is label-guided selection and is shown only to locate the paper value inside the run-to-run spread.

Kaggle settings: GPU on, Internet on, no input dataset. Code is cloned from GitHub at a pinned commit.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PROTOCOL = "mvdec_2025_scratch_both_views_true_bottleneck_min500_release_l2_v1"
PINNED_COMMIT = "b3d2071ba238ad2e2210573ab1133799272bb319"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/working/mvdec-s2-10seeds-reuters")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path in (REPO, OUTPUT):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", "dev", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)


In [ ]:
for seed in range(42, 52):
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", "REUTERS",
        "--runs", "1", "--seed", str(seed),
        "--protocol", PROTOCOL,
        "--progress-interval", "100",
        "--dataset-root", "external_repos/DEKM/datasets",
        "--public-output-dir", str(OUTPUT),
    ]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True, timeout=4 * 60 * 60)


In [ ]:
import statistics

PAPER = {'view1': (0.7642, 0.5999), 'view2': (0.6501, 0.5129), 'fused': (0.7813, 0.6141)}
manifests = sorted(OUTPUT.rglob("manifest.json"))
assert len(manifests) == 10, len(manifests)
rows = []
for path in manifests:
    manifest = json.loads(path.read_text(encoding="utf-8"))
    assert manifest["status"] == "complete" and manifest["protocol_id"] == PROTOCOL
    rows.append((manifest["seed"], manifest["metrics"]))
rows.sort(key=lambda row: row[0])

print("seed  " + "  ".join(key + "_acc/nmi" for key in PAPER))
for seed, metrics in rows:
    print(seed, "  ".join("%.4f/%.4f" % (metrics[key + "_acc"], metrics[key + "_nmi"]) for key in PAPER))
print()
print("rep    mean_ACC    std  median   best | paper_ACC | mean_NMI best_NMI | paper_NMI")
for key, (paper_acc, paper_nmi) in PAPER.items():
    accs = [metrics[key + "_acc"] for _, metrics in rows]
    nmis = [metrics[key + "_nmi"] for _, metrics in rows]
    print("%-6s %8.4f %6.4f %7.4f %7.4f | %.4f | %.4f %.4f | %.4f" % (key, statistics.mean(accs), statistics.stdev(accs), statistics.median(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

zip_path = Path(shutil.make_archive("/kaggle/working/mvdec-s2-10seeds-reuters-results", "zip", OUTPUT.parent, OUTPUT.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)


In [ ]:
for module, extra in (
    ("pipeline.mvdec_audit", ["--require-all-complete"]),
):
    subprocess.run(
        [
            sys.executable, "-m", module,
            "--root", str(OUTPUT),
            "--expected-datasets", "REUTERS",
            "--expected-protocol-id", PROTOCOL,
            "--expected-training-commit", TRAINING_COMMIT,
            *extra,
            *(["--dataset-root", "external_repos/DEKM/datasets"] if module.endswith("mvdec_audit") else []),
        ],
        cwd=REPO,
        check=True,
    )